# Reaction–diffusion–advection: parametric dataset generation

Paired **HF** (spatial PDE, FEniCSx) and **LF** (lumped ODE) trajectories over an explicit 3-parameter family.

Both fidelities see the same changes in **linear kinetics** and **temporal source forcing**; HF additionally resolves spatial transport and source profiles $S_i(\mathbf{x})$.

## Fixed vs. varied physics

**Fixed for every case**
- Initial conditions (6 species)
- Spatial source profiles $S_i(\mathbf{x})$ on species **0, 2, 4** only (species 1, 3, 5 have zero spatial source)
- Quadratic reaction network (8 reactions + one extra $c_2+c_4\to c_5$ link)
- Diffusion coefficients and advection (`velocity_scale = 5`, double-gyre)

**Varied (3 parameters, Cartesian product of the lists below)**

| Parameter | Meaning | HF & LF |
|-----------|---------|---------|
| `linear_scale` | Scales columns 0–2 of the linear rate matrix (kinetics of species 0, 1, 2). Mass balance among species 0–4 is preserved at any scale. | both |
| `source_t_peak` | Centre time of the three source pulses (species 0, 2, 4) | both |
| `source_pulse_width` | Width of all three pulses (Gaussian, sine bump, skewed exponential) | both |

**Temporal source shapes (species 0, 2, 4)**
- $c_0$: symmetric Gaussian $\exp[-\tfrac12((t-t_\mathrm{peak})/w)^2]$
- $c_2$: compact sine-squared bump on $[t_\mathrm{peak}-w,\, t_\mathrm{peak}+w]$
- $c_4$: ramp up then exponential decay (skewed pulse)

## Simulation settings

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from models.dataset import (
    BASE_LINEAR_REACTION,
    DatasetConfig,
    build_param_grid,
    generate_dataset,
    save_dataset,
    st_lambda_profiles,
)

N = 32
num_species = 6
dt = 0.01
T_final = 10.0
save_snapshots_every = 2
velocity_scale = 5.0

output_dir = Path("data/reaction_diffusion/rd_kinetics_sources")

config = DatasetConfig(
    N=N,
    num_species=num_species,
    dt=dt,
    T_final=T_final,
    save_every=save_snapshots_every,
    velocity_scale=velocity_scale,
)

## Linear reaction matrix (reference at `linear_scale = 1`)

Columns 0–4 sum to zero (mass redistributed). Column 5 is the terminal pool.

In [ ]:
pd.DataFrame(
    BASE_LINEAR_REACTION,
    columns=[f"c{j}" for j in range(num_species)],
    index=[f"eq{i}" for i in range(num_species)],
).style.format("{:.2f}")

## Parameter selection

In [ ]:
# linear_scales = np.linspace(0.5, 2, 20)
# source_t_peaks = np.linspace(0.5, 3, 5)
# source_pulse_widths = np.linspace(0.2, 2.0, 10)
linear_scales = np.linspace(0.5, 2, 10)
source_t_peaks = np.linspace(0.5, 3, 5)
source_pulse_widths = np.linspace(0.2, 2.0, 10)

params = build_param_grid(linear_scales, source_t_peaks, source_pulse_widths)

param_table = pd.DataFrame(
    params,
    columns=["linear_scale", "source_t_peak", "source_pulse_width"],
)
param_table.index.name = "case"
param_table

## Preview temporal sources for one parameter choice

Pick a row from the table above to inspect the three distinct pulse shapes on species 0, 2, 4.

In [ ]:
preview_case = 0
t_peak, width = params[preview_case, 1], params[preview_case, 2]
preview_times = np.linspace(0, 10, 400)
profiles = st_lambda_profiles(t_peak, width, preview_times, num_species)

fig, axs = plt.subplots(1, 3, figsize=(15, 3))  
for species, ax in zip((0, 2, 4), axs):
    ax.plot(preview_times, profiles[species], label=f"species {species}")
    ax.set_xlabel("Time")
    ax.set_ylabel(f"St_lambda_{species}")
    ax.set_title(f"case {preview_case}: t_peak={t_peak:g}, width={width:g}")
plt.tight_layout()

## Run HF and LF for all cases

In [ ]:
data = generate_dataset(config, params, progress=True)

print(f"Nparams = {data['params'].shape[0]}")
print(f"Nt      = {data['times'].shape[0]}")
print(f"Nspace  = {data['nodes'].shape[0]}")
print("HF per species : (Nparams, Nt, Nspace)")
print("LF             : (Nparams, Nt, num_species)")
print(f"HF solve time  : mean={data['timing_hf'].mean():.2f}s, std={data['timing_hf'].std():.2f}s")
print(f"LF solve time  : mean={data['timing_lf'].mean():.3f}s, std={data['timing_lf'].std():.3f}s")

## Save to disk

| File | `data` shape |
|------|-------------|
| `mesh.npz` | `nodes` → `(Nspace, 2)` |
| `params.npz` | metadata + `(Nparams, 3)` parameter table |
| `lf.npz` | `(Nparams, Nt, num_species)` |
| `hf_c{i}.npz` | `(Nparams, Nt, Nspace)` per species |

In [ ]:
saved_dir = save_dataset(data, output_dir)
print(f"Saved to {saved_dir}")
print("Files:", sorted(p.name for p in saved_dir.glob("*.npz")))

## Sanity check: HF spatial mean vs LF

Both curves should move with the parameter case. Differences highlight the LF closure gap (spatial sources and quadratic reactions).

In [ ]:
case_idx = 485

ls, tp, w = data["params"][case_idx]
nrows = 2
ncols = num_species // nrows
fig, axs = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*3))
axs = axs.ravel()

for species_idx in range(num_species):
    ax = axs[species_idx]
    hf_mean = data["hf"][case_idx, species_idx].mean(axis=-1)
    lf_series = data["lf"][case_idx, :, species_idx]

    ax.plot(data["times"], hf_mean, label="HF spatial mean")
    ax.plot(data["times"], lf_series, "--", label="LF (ODE)")
    ax.set_xlabel("Time")
    ax.set_ylabel(f"Species c_{species_idx}")
fig.suptitle(
    f"case {case_idx}: linear_scale={ls:g}, "
    f"t_peak={tp:g}, width={w:g}"
)

_lin, _lab = ax.get_legend_handles_labels()
fig.legend(_lin, _lab, loc="upper left")
plt.tight_layout()

## Reload example

In [ ]:
mesh = np.load(saved_dir / "mesh.npz")
meta = np.load(saved_dir / "params.npz")
lf = np.load(saved_dir / "lf.npz")
hf0 = np.load(saved_dir / "hf_c0.npz")

print("mesh nodes:", mesh["nodes"].shape)
print("params:", meta["params"].shape)
print("lf:", lf["data"].shape)
print("hf_c0:", hf0["data"].shape)

## Solve time comparison

Wall-clock time per parameter case:
- **HF:** full `advance` loop (all time steps, including snapshot storage)
- **LF:** `solve_ivp` integration over `[0, T_final]`

Timings are stored in `params.npz` as `timing_hf` and `timing_lf` (seconds, one value per case).

In [ ]:
timing_table = param_table.copy()
timing_table["timing_hf_s"] = data["timing_hf"]
timing_table["timing_lf_s"] = data["timing_lf"]
timing_table["hf_over_lf"] = data["timing_hf"] / np.maximum(data["timing_lf"], 1e-12)
timing_table

In [ ]:
labels = ["HF (PDE advance loop)", "LF (ODE solve_ivp)"]
means = [data["timing_hf"].mean(), data["timing_lf"].mean()]
stds = [data["timing_hf"].std(), data["timing_lf"].std()]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, means, yerr=stds, capsize=6, color=["#1f77b4", "#ff7f0e"], alpha=0.85)
ax.set_ylabel("Wall time [s]")
ax.set_title(f"Mean solve time over {len(data['timing_hf'])} parameter cases")

ax.set_yscale('log')

plt.tight_layout()